# RAG Pipeline — Gemini Q&A over Vertex AI Documentation

Builds a full Retrieval-Augmented Generation pipeline that answers questions about
Google Cloud / Vertex AI using the official documentation as its knowledge base.

**Backend toggle** — set once here, used throughout the notebook:
- `"gemini_api"` — free Gemini API (Days 2–3 development)
- `"vertex_ai"` — enterprise Vertex AI SDK (Day 4 verification)

**Sections:**
1. Setup & Auth
2. Corpus Ingestion
3. Chunking
4. Embedding
5. Retrieval
6. Generation
7. Interactive Q&A ← demo centerpiece
8. RAGAS Evaluation _(Day 3)_
9. Analysis _(Day 3)_

## Section 1 — Setup & Auth

In [ ]:
import os
import sys
import time
import json
import subprocess
from pathlib import Path

import numpy as np
from dotenv import load_dotenv

# ── Backend toggle ─────────────────────────────────────────────────────────────
BACKEND = "gemini_api"   # or "vertex_ai"
# ───────────────────────────────────────────────────────────────────────────────

load_dotenv(dotenv_path="../.env")
sys.path.insert(0, str(Path("../src").resolve()))

GENERATION_MODEL = "gemini-2.5-flash"
EMBEDDING_MODEL  = "gemini-embedding-001"

if BACKEND == "gemini_api":
    from google import genai
    from google.genai import errors as genai_errors

    api_key = os.environ.get("GEMINI_API_KEY")
    assert api_key, "GEMINI_API_KEY not set — check .env"
    client = genai.Client(api_key=api_key)
    print(f"Backend  : Gemini API (free tier)")
    print(f"API key  : {api_key[:8]}...")

elif BACKEND == "vertex_ai":
    import vertexai
    from vertexai.generative_models import GenerativeModel
    from google.api_core import exceptions as gcp_errors

    project = os.environ.get("GCP_PROJECT_ID")
    location = os.environ.get("GCP_LOCATION", "us-central1")
    assert project, "GCP_PROJECT_ID not set — check .env"
    vertexai.init(project=project, location=location)
    client = None  # Vertex AI uses module-level calls
    print(f"Backend  : Vertex AI")
    print(f"Project  : {project}  Location: {location}")

else:
    raise ValueError(f"Unknown BACKEND: {BACKEND!r}. Use 'gemini_api' or 'vertex_ai'.")

print(f"Generation model : {GENERATION_MODEL}")
print(f"Embedding model  : {EMBEDDING_MODEL}")

In [ ]:
def with_retry(fn, retries=4, base_delay=5):
    """Retry fn on 503 UNAVAILABLE with exponential backoff."""
    for attempt in range(retries):
        try:
            return fn()
        except Exception as e:
            if "503" not in str(e) and "UNAVAILABLE" not in str(e):
                raise
            if attempt == retries - 1:
                raise
            delay = base_delay * (2 ** attempt)
            print(f"503 UNAVAILABLE — retrying in {delay}s (attempt {attempt + 1}/{retries})...")
            time.sleep(delay)

print("Retry helper ready.")

## Section 2 — Corpus Ingestion

Fetches and cleans GCP/Vertex AI documentation pages into `corpus/`.
The script (`src/build_corpus.py`) is idempotent — safe to re-run, skips nothing
already saved. Expect 15–30 seconds to fetch all pages with polite crawl delays.

Skip this cell if `corpus/manifest.json` already exists from a previous run.

In [ ]:
CORPUS_DIR = Path("../corpus")
MANIFEST_PATH = CORPUS_DIR / "manifest.json"

if MANIFEST_PATH.exists():
    print("Corpus already built — loading manifest.")
else:
    print("Building corpus (this takes ~30s) ...")
    result = subprocess.run(
        [sys.executable, "../src/build_corpus.py"],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise RuntimeError("build_corpus.py failed")

manifest = json.loads(MANIFEST_PATH.read_text())
pages = manifest["pages"]
skipped = manifest.get("skipped", [])

print(f"\nCorpus: {len(pages)} pages loaded, {len(skipped)} skipped")
for p in pages:
    print(f"  {p['slug']:<40} {p['chars']:>8,} chars")

## Section 3 — Chunking

_Issue #7 — coming next._

In [ ]:
# TODO (#7): implement chunker and display sample chunks
print("Section 3 not yet implemented — see issue #7")

## Section 4 — Embedding

_Issue #8 — coming next._

In [ ]:
# TODO (#8): embed chunks and persist to corpus/
print("Section 4 not yet implemented — see issue #8")

## Section 5 — Retrieval

_Issue #9 — coming next._

In [ ]:
# TODO (#9): cosine similarity retrieval
print("Section 5 not yet implemented — see issue #9")

## Section 6 — Generation

_Issue #10 — coming next._

In [ ]:
# TODO (#10): RAG-augmented Gemini generation
print("Section 6 not yet implemented — see issue #10")

## Section 7 — Interactive Q&A

_Issue #11 — coming next. This is the demo centerpiece._

In [ ]:
# TODO (#11): ask(question) → retrieved chunks + generated answer
print("Section 7 not yet implemented — see issue #11")

## Section 8 — RAGAS Evaluation

_Issues #12–13 — Day 3._

In [ ]:
# TODO (#12, #13): RAGAS evaluation over test set
print("Section 8 not yet implemented — see issues #12–13")

## Section 9 — Analysis

_Issue #14 — Day 3 notebook polish._